In [1]:
import numpy as np
rel_path = '../'

In [2]:
def clip(input, qbit):
  max_value = 2. ** (qbit-1) -1.
  min_value = -2. ** (qbit-1)
  output = np.clip(input, min_value, max_value)
  return output

def uniform_quantize(input, qbit):
  min_value = min(input)
  max_value = max(input)
  abs_min_value = abs(min_value)
  abs_max_value = abs(max_value)
  if (abs_max_value >= abs_min_value):
    min_cond = -abs_max_value
    max_cond = abs_max_value
  else:
    min_cond = -abs_min_value
    max_cond = abs_min_value
  qmin = 0
  qmax = 2. ** qbit - 1.
  scale = (max_cond - min_cond) / (qmax - qmin)
  output = input / scale
  output = np.floor(output)
  output = clip(input=output, qbit=qbit)
  return output, scale

In [3]:
wdir = rel_path + 'tmpdata/'

fp_w0 = wdir + 'w0.fc1.weight.csv'
fp_w0_scale = wdir + 'w0.fc1.weight_scale.csv'
np_w0 = np.loadtxt(fp_w0, delimiter=',', dtype=float)
np_w0_scale = np.loadtxt(fp_w0_scale, delimiter=',', dtype=float)

fp_w2 = wdir + 'w2.fc2.weight.csv'
fp_w2_scale = wdir + 'w2.fc2.weight_scale.csv'
np_w2 = np.loadtxt(fp_w2, delimiter=',', dtype=float)
np_w2_scale = np.loadtxt(fp_w2_scale, delimiter=',', dtype=float)


In [4]:
# ol?

In [5]:
# PYNQ-Z2 Initialize
from pynq import Overlay, Interrupt
ol = Overlay("./design_top.bit")
top = ol.top_0

def dec_to_tc(data, bit=8):
    if data >= 0:
        return data
    else:
        datatc = pow(2,bit-1) + (pow(2,bit-1)+data)
        return int(datatc)
def tc_to_dec(data, bit=32):
    if data >= pow(2,bit-1):
        datadec = data-2*pow(2,bit-1)
        return int(datadec)
    else:
        return data
def instr_param(opvalid, opcode, param, data):
    _instr = opvalid * pow(2,31)
    _instr += opcode * pow(2,28)
    _instr += param * pow(2,8)
    _instr += data * pow(2,0)
    return int(_instr)    

def instr_data(opvalid, opcode, sel, addr, data):
    _instr = opvalid * pow(2,31)
    _instr += opcode * pow(2,28)
    _instr += sel * pow(2,24)
    _instr += addr * pow(2,8)
#     _instr += data * pow(2,0)
    _instr += dec_to_tc(data)
    return int(_instr)
def pl_rst():
    top.mmio.write(offset=0, data=dec_to_tc(data=-1,bit=32))
    top.mmio.write(offset=0, data=0)
def finish_check():
    mmio_read = top.mmio.read(offset=4)
    read_data = mmio_read / np.power(2,16) / np.power(2,15)
    return int(read_data)
    
# parameters setting
OPCODE_NOP = 0
OPCODE_PARAM = 1
OPCODE_LDSRAM = 2
OPCODE_STSRAM = 3
OPCODE_EX = 4
OPCODE_WBPSRAM = 5
OPCODE_WBPARAM = 6

PARAM_BASE_WSRAM = 0
PARAM_S = 1
PARAM_OC = 2
PARAM_IC = 3
PARAM_TRG = 4

PARAM_IC_WH = 5
PARAM_BASE_WSRAM_WH = 6

TRG_ISRAM = 0
TRG_WSRAM = 1
TRG_PSRAM = 2

In [6]:
# Load np_w0
baseaddr = 0
top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_TRG, data=TRG_WSRAM))
for i in range(128):
    for j in range(784):
        x = i % 8
        y = baseaddr + int(np.floor(i/8)) * 784 + j
        top.mmio.write(offset=0, data=instr_data(opvalid=1, opcode=OPCODE_LDSRAM, sel=x, addr=y, data=np_w0[i*784+j]))

In [7]:
# Load np_w2
baseaddr = int(128/8*784)
top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_TRG, data=TRG_WSRAM))
for i in range(10):
    for j in range(128):
        x = i % 8
        y = baseaddr + int(np.floor(i/8)) * 128 + j
        top.mmio.write(offset=0, data=instr_data(opvalid=1, opcode=OPCODE_LDSRAM, sel=x, addr=y, data=np_w2[i*128+j]))

In [8]:
# Parameter Test
top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_NOP, param=0, data=0))
top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_S, data=1))
top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_OC, data=128))
top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_IC, data=784%256))
top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_IC_WH, data=int(784/256)))
top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_WBPARAM, param=PARAM_S, data=0))
print(top.mmio.read(offset=12))
top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_WBPARAM, param=PARAM_OC, data=0))
print(top.mmio.read(offset=12))
top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_WBPARAM, param=PARAM_IC, data=0))
print(top.mmio.read(offset=12))

1
128
784


In [9]:
wdir = rel_path + 'tmpdata/'
fp_input_csv = wdir + 'mnist_test.csv'
np_input_csv = np.loadtxt(fp_input_csv, delimiter=',', dtype=float)

print(np_input_csv.shape)

from pynq import Clocks
Clocks.fclk0_mhz

(10000, 785)


100.0

In [10]:
testsize = 9999
correct = 0
count = 0
for b in range(testsize):
  _label = int(np_input_csv[b][0])
  _image = np_input_csv[b][1:785]

  # fc1
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_OC, data=128))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_IC, data=784%256))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_IC_WH, data=int(784/256)))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM, data=0))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM_WH, data=0))
#   top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_WBPARAM, param=PARAM_S, data=0))
#   print(top.mmio.read(offset=12))
#   top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_WBPARAM, param=PARAM_OC, data=0))
#   print(top.mmio.read(offset=12))
#   top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_WBPARAM, param=PARAM_IC, data=0))
#   print(top.mmio.read(offset=12))

  np_ia0_q, scale_ia0 = uniform_quantize(input=_image, qbit=8)  
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_TRG, data=TRG_ISRAM))
  for i in range(784):
    x = int(i/8)
    y = i % 8
    top.mmio.write(offset=0, data=instr_data(opvalid=1, opcode=OPCODE_LDSRAM, sel=y, addr=x, data=np_ia0_q[i]))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM, data=0))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_EX, param=0, data=0))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_NOP, param=0, data=0))
  np_oa0 = np.zeros([128], dtype=float)

  while not finish_check():
    pass
  for i in range(int(np.ceil(128/8))):
    for j in range(8):
      if j+8*i < 128:
        top.mmio.write(offset=0, data=instr_data(opvalid=1, opcode=OPCODE_WBPSRAM, sel=j, addr=i, data=0))
        np_oa0[j+8*i]=tc_to_dec(data=top.mmio.read(offset=12),bit=32)
        np_oa0[j+8*i]*= scale_ia0 * np_w0_scale
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_NOP, param=0, data=0))

  
#   #fc2
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_OC, data=10))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_IC, data=128))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_IC_WH, data=0))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM, data=int(128/8*784)))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM_WH, data=int(128/8*784/256)))
  np_ia2 = np.empty([128], dtype=float)
  for i in range(128):
    np_ia2[i] = np_oa0[i] if np_oa0[i] > 0 else 0


  np_ia2_q, scale_ia2 = uniform_quantize(input=np_ia2, qbit=8)

  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_TRG, data=TRG_ISRAM))
  for i in range(128):
    x = int(i/8)
    y = i % 8
    top.mmio.write(offset=0, data=instr_data(opvalid=1, opcode=OPCODE_LDSRAM, sel=y, addr=x, data=np_ia2_q[i]))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM, data=int(128/8*784)))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_PARAM, param=PARAM_BASE_WSRAM_WH, data=int(128/8*784/256)))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_EX, param=0, data=0))
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_NOP, param=0, data=0))
  np_oa2 = np.zeros([10], dtype=float)

  while not finish_check():
    pass
  for i in range(int(np.ceil(10/8))):
    for j in range(8):
      if j+8*i < 10:
        top.mmio.write(offset=0, data=instr_data(opvalid=1, opcode=OPCODE_WBPSRAM, sel=j, addr=i, data=0))
        np_oa2[j+8*i]=tc_to_dec(data=top.mmio.read(offset=12),bit=32)
        np_oa2[j+8*i]*= scale_ia2 * np_w2_scale
  top.mmio.write(offset=0, data=instr_param(opvalid=1, opcode=OPCODE_NOP, param=0, data=0))
  _infer = np.argmax(np_oa2)
  count += 1
  if _infer == _label:
    correct += 1
  print("inference: " + str(_infer) + ", answer: " + str(_label)) 
  print(f"Count: {count}, Correct: {correct}, Accuracy: {correct/count*100}%\n")

inference: 7, answer: 7
Count: 1, Correct: 1, Accuracy: 100.0%

inference: 2, answer: 2
Count: 2, Correct: 2, Accuracy: 100.0%

inference: 1, answer: 1
Count: 3, Correct: 3, Accuracy: 100.0%

inference: 0, answer: 0
Count: 4, Correct: 4, Accuracy: 100.0%

inference: 4, answer: 4
Count: 5, Correct: 5, Accuracy: 100.0%

inference: 1, answer: 1
Count: 6, Correct: 6, Accuracy: 100.0%

inference: 4, answer: 4
Count: 7, Correct: 7, Accuracy: 100.0%

inference: 9, answer: 9
Count: 8, Correct: 8, Accuracy: 100.0%

inference: 0, answer: 5
Count: 9, Correct: 8, Accuracy: 88.88888888888889%

inference: 9, answer: 9
Count: 10, Correct: 9, Accuracy: 90.0%

inference: 0, answer: 0
Count: 11, Correct: 10, Accuracy: 90.9090909090909%

inference: 6, answer: 6
Count: 12, Correct: 11, Accuracy: 91.66666666666666%

inference: 9, answer: 9
Count: 13, Correct: 12, Accuracy: 92.3076923076923%

inference: 0, answer: 0
Count: 14, Correct: 13, Accuracy: 92.85714285714286%

inference: 1, answer: 1
Count: 15, Cor

inference: 3, answer: 7
Count: 112, Correct: 102, Accuracy: 91.07142857142857%

inference: 3, answer: 3
Count: 113, Correct: 103, Accuracy: 91.1504424778761%

inference: 9, answer: 9
Count: 114, Correct: 104, Accuracy: 91.22807017543859%

inference: 7, answer: 7
Count: 115, Correct: 105, Accuracy: 91.30434782608695%

inference: 9, answer: 4
Count: 116, Correct: 105, Accuracy: 90.51724137931035%

inference: 4, answer: 4
Count: 117, Correct: 106, Accuracy: 90.5982905982906%

inference: 4, answer: 4
Count: 118, Correct: 107, Accuracy: 90.67796610169492%

inference: 9, answer: 9
Count: 119, Correct: 108, Accuracy: 90.75630252100841%

inference: 2, answer: 2
Count: 120, Correct: 109, Accuracy: 90.83333333333333%

inference: 3, answer: 5
Count: 121, Correct: 109, Accuracy: 90.08264462809917%

inference: 4, answer: 4
Count: 122, Correct: 110, Accuracy: 90.1639344262295%

inference: 7, answer: 7
Count: 123, Correct: 111, Accuracy: 90.2439024390244%

inference: 6, answer: 6
Count: 124, Correct:

inference: 0, answer: 0
Count: 216, Correct: 193, Accuracy: 89.35185185185185%

inference: 3, answer: 3
Count: 217, Correct: 194, Accuracy: 89.40092165898618%

inference: 6, answer: 6
Count: 218, Correct: 195, Accuracy: 89.44954128440367%

inference: 5, answer: 5
Count: 219, Correct: 196, Accuracy: 89.49771689497716%

inference: 3, answer: 5
Count: 220, Correct: 196, Accuracy: 89.0909090909091%

inference: 7, answer: 7
Count: 221, Correct: 197, Accuracy: 89.14027149321268%

inference: 2, answer: 2
Count: 222, Correct: 198, Accuracy: 89.1891891891892%

inference: 2, answer: 2
Count: 223, Correct: 199, Accuracy: 89.23766816143498%

inference: 7, answer: 7
Count: 224, Correct: 200, Accuracy: 89.28571428571429%

inference: 1, answer: 1
Count: 225, Correct: 201, Accuracy: 89.33333333333333%

inference: 2, answer: 2
Count: 226, Correct: 202, Accuracy: 89.38053097345133%

inference: 8, answer: 8
Count: 227, Correct: 203, Accuracy: 89.42731277533039%

inference: 4, answer: 4
Count: 228, Correc

inference: 9, answer: 9
Count: 321, Correct: 283, Accuracy: 88.1619937694704%

inference: 3, answer: 2
Count: 322, Correct: 283, Accuracy: 87.88819875776397%

inference: 9, answer: 9
Count: 323, Correct: 284, Accuracy: 87.92569659442725%

inference: 3, answer: 3
Count: 324, Correct: 285, Accuracy: 87.96296296296296%

inference: 0, answer: 0
Count: 325, Correct: 286, Accuracy: 88.0%

inference: 4, answer: 4
Count: 326, Correct: 287, Accuracy: 88.03680981595092%

inference: 2, answer: 2
Count: 327, Correct: 288, Accuracy: 88.07339449541286%

inference: 0, answer: 0
Count: 328, Correct: 289, Accuracy: 88.10975609756098%

inference: 7, answer: 7
Count: 329, Correct: 290, Accuracy: 88.14589665653494%

inference: 1, answer: 1
Count: 330, Correct: 291, Accuracy: 88.18181818181819%

inference: 1, answer: 1
Count: 331, Correct: 292, Accuracy: 88.21752265861026%

inference: 2, answer: 2
Count: 332, Correct: 293, Accuracy: 88.25301204819277%

inference: 1, answer: 1
Count: 333, Correct: 294, Accu

inference: 0, answer: 0
Count: 425, Correct: 376, Accuracy: 88.47058823529412%

inference: 4, answer: 4
Count: 426, Correct: 377, Accuracy: 88.49765258215963%

inference: 9, answer: 9
Count: 427, Correct: 378, Accuracy: 88.52459016393442%

inference: 1, answer: 1
Count: 428, Correct: 379, Accuracy: 88.55140186915888%

inference: 4, answer: 4
Count: 429, Correct: 380, Accuracy: 88.57808857808858%

inference: 8, answer: 8
Count: 430, Correct: 381, Accuracy: 88.60465116279069%

inference: 1, answer: 1
Count: 431, Correct: 382, Accuracy: 88.63109048723898%

inference: 8, answer: 8
Count: 432, Correct: 383, Accuracy: 88.6574074074074%

inference: 4, answer: 4
Count: 433, Correct: 384, Accuracy: 88.68360277136259%

inference: 8, answer: 5
Count: 434, Correct: 384, Accuracy: 88.47926267281106%

inference: 9, answer: 9
Count: 435, Correct: 385, Accuracy: 88.50574712643679%

inference: 3, answer: 8
Count: 436, Correct: 385, Accuracy: 88.30275229357798%

inference: 8, answer: 8
Count: 437, Corre

...

inference: 0, answer: 0
Count: 9509, Correct: 8449, Accuracy: 88.85266589546745%

inference: 1, answer: 1
Count: 9510, Correct: 8450, Accuracy: 88.85383806519454%

inference: 2, answer: 2
Count: 9511, Correct: 8451, Accuracy: 88.85500998843445%

inference: 3, answer: 3
Count: 9512, Correct: 8452, Accuracy: 88.85618166526493%

inference: 4, answer: 4
Count: 9513, Correct: 8453, Accuracy: 88.85735309576368%

inference: 8, answer: 5
Count: 9514, Correct: 8453, Accuracy: 88.84801345385748%

inference: 6, answer: 6
Count: 9515, Correct: 8454, Accuracy: 88.84918549658434%

inference: 7, answer: 7
Count: 9516, Correct: 8455, Accuracy: 88.85035729298025%

inference: 8, answer: 8
Count: 9517, Correct: 8456, Accuracy: 88.85152884312284%

inference: 9, answer: 9
Count: 9518, Correct: 8457, Accuracy: 88.85270014708972%

inference: 1, answer: 1
Count: 9519, Correct: 8458, Accuracy: 88.8538712049585%

inference: 0, answer: 0
Count: 9520, Correct: 8459, Accuracy: 88.85504201680672%

inference: 1, ans

inference: 9, answer: 9
Count: 9611, Correct: 8542, Accuracy: 88.87732806159609%

inference: 0, answer: 0
Count: 9612, Correct: 8543, Accuracy: 88.87848522679982%

inference: 1, answer: 1
Count: 9613, Correct: 8544, Accuracy: 88.8796421512535%

inference: 2, answer: 2
Count: 9614, Correct: 8545, Accuracy: 88.88079883503225%

inference: 3, answer: 3
Count: 9615, Correct: 8546, Accuracy: 88.88195527821114%

inference: 4, answer: 4
Count: 9616, Correct: 8547, Accuracy: 88.88311148086522%

inference: 5, answer: 5
Count: 9617, Correct: 8548, Accuracy: 88.88426744306956%

inference: 6, answer: 6
Count: 9618, Correct: 8549, Accuracy: 88.88542316489915%

inference: 7, answer: 7
Count: 9619, Correct: 8550, Accuracy: 88.88657864642894%

inference: 8, answer: 8
Count: 9620, Correct: 8551, Accuracy: 88.88773388773389%

inference: 9, answer: 9
Count: 9621, Correct: 8552, Accuracy: 88.88888888888889%

inference: 0, answer: 0
Count: 9622, Correct: 8553, Accuracy: 88.89004364996882%

inference: 1, ans

inference: 7, answer: 7
Count: 9712, Correct: 8634, Accuracy: 88.9003294892916%

inference: 8, answer: 8
Count: 9713, Correct: 8635, Accuracy: 88.90147225368064%

inference: 9, answer: 9
Count: 9714, Correct: 8636, Accuracy: 88.90261478278772%

inference: 0, answer: 0
Count: 9715, Correct: 8637, Accuracy: 88.90375707668554%

inference: 1, answer: 1
Count: 9716, Correct: 8638, Accuracy: 88.9048991354467%

inference: 2, answer: 2
Count: 9717, Correct: 8639, Accuracy: 88.90604095914377%

inference: 3, answer: 3
Count: 9718, Correct: 8640, Accuracy: 88.90718254784935%

inference: 4, answer: 4
Count: 9719, Correct: 8641, Accuracy: 88.90832390163597%

inference: 0, answer: 5
Count: 9720, Correct: 8641, Accuracy: 88.89917695473251%

inference: 0, answer: 6
Count: 9721, Correct: 8641, Accuracy: 88.89003188972328%

inference: 7, answer: 7
Count: 9722, Correct: 8642, Accuracy: 88.89117465542068%

inference: 8, answer: 8
Count: 9723, Correct: 8643, Accuracy: 88.89231718605369%

inference: 9, answ

inference: 4, answer: 4
Count: 9814, Correct: 8712, Accuracy: 88.77114326472386%

inference: 0, answer: 5
Count: 9815, Correct: 8712, Accuracy: 88.762098828324%

inference: 6, answer: 6
Count: 9816, Correct: 8713, Accuracy: 88.76324368378158%

inference: 7, answer: 7
Count: 9817, Correct: 8714, Accuracy: 88.7643883059998%

inference: 8, answer: 8
Count: 9818, Correct: 8715, Accuracy: 88.76553269504991%

inference: 0, answer: 0
Count: 9819, Correct: 8716, Accuracy: 88.76667685100315%

inference: 1, answer: 1
Count: 9820, Correct: 8717, Accuracy: 88.76782077393075%

inference: 2, answer: 2
Count: 9821, Correct: 8718, Accuracy: 88.76896446390387%

inference: 3, answer: 3
Count: 9822, Correct: 8719, Accuracy: 88.7701079209937%

inference: 4, answer: 4
Count: 9823, Correct: 8720, Accuracy: 88.77125114527131%

inference: 7, answer: 7
Count: 9824, Correct: 8721, Accuracy: 88.77239413680782%

inference: 8, answer: 8
Count: 9825, Correct: 8722, Accuracy: 88.7735368956743%

inference: 9, answer:

inference: 3, answer: 3
Count: 9915, Correct: 8799, Accuracy: 88.74432677760969%

inference: 4, answer: 4
Count: 9916, Correct: 8800, Accuracy: 88.74546187979023%

inference: 9, answer: 7
Count: 9917, Correct: 8800, Accuracy: 88.73651305838459%

inference: 8, answer: 8
Count: 9918, Correct: 8801, Accuracy: 88.7376487194999%

inference: 9, answer: 9
Count: 9919, Correct: 8802, Accuracy: 88.7387841516282%

inference: 7, answer: 7
Count: 9920, Correct: 8803, Accuracy: 88.73991935483872%

inference: 8, answer: 8
Count: 9921, Correct: 8804, Accuracy: 88.74105432920068%

inference: 6, answer: 6
Count: 9922, Correct: 8805, Accuracy: 88.74218907478331%

inference: 4, answer: 4
Count: 9923, Correct: 8806, Accuracy: 88.74332359165575%

inference: 1, answer: 1
Count: 9924, Correct: 8807, Accuracy: 88.74445787988714%

inference: 9, answer: 9
Count: 9925, Correct: 8808, Accuracy: 88.7455919395466%

inference: 3, answer: 3
Count: 9926, Correct: 8809, Accuracy: 88.74672577070321%

inference: 8, answe